# 🤖 RS AI — Colab Launcher (සිංහල/English chatbot server)

නිල වශයෙන් මේ notebook එක:
1. ඔයාගේ RS AI server එක **Colab (free GPU) එකේ** run කරනවා
2. **Public URL** එකක් හදනවා (Cloudflare tunnel — free)
3. ඒ URL එක **Android app එකට / web UI එකට / ඕනම client එකකට** දාන්න පුළුවන්

**Smart mode** (thinking 🔬📷 vision) ඕන නම් — [Groq free API key](https://console.groq.com/keys) එකක් cell 3 වල දාන්න.
කළු දකුණෙන් **Runtime → Run all** තියෙනවා 👇

In [ ]:
# 1️⃣ Dependencies install (≈ 2 min)
%pip install --quiet torch==2.2.2 sentencepiece fastapi "uvicorn[standard]" requests "numpy<2" pypdf
print("✅ dependencies installed")

In [ ]:
# 2️⃣ Repo clone
import os
if not os.path.exists("/content/Rs-et"):
    !git clone --depth 1 https://github.com/Rusindu12/Rs-et.git /content/Rs-et
os.chdir("/content/Rs-et")
print("✅ repo ready:", os.getcwd())

In [ ]:
# 3️⃣ (optional) SMART MODE — Groq free key දාන්න: https://console.groq.com/keys
import os
os.environ["GROQ_API_KEY"] = ""        # "gsk_..." here = thinking/vision/research unlocked
# os.environ["RS_API_TOKEN"] = ""      # public security token (recommend for public links)
print("smart mode:", "ON ⚡" if os.environ.get("GROQ_API_KEY") else "OFF (local RS-GPT only)")

In [ ]:
# 4️⃣ Server start + Cloudflare public URL
import subprocess, time, re, threading, os

server = subprocess.Popen(
    ["python3", "-m", "uvicorn", "server.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
time.sleep(12)
print(open("/proc/%s/fd/1" % server.pid).read() if False else "server starting (pid %s)" % server.pid)

!wget -qnc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
def watcher():
    global public_url
    for line in proc.stdout:
        m = re.search(r"https://[-a-zA-Z0-9.]+\.trycloudflare\.com", line)
        if m and not public_url:
            public_url = m.group(0)
            print("\n" + "=" * 60)
            print("🌍 PUBLIC URL:", public_url)
            print("=" * 60)
            print("➡️ Android app: Settings ⚙️ → server URL = මේක")
            print("➡️ Web UI:", public_url)
            print("➡️ /diagnose:", public_url + "/diagnose")
            print("➡️ Any app:", public_url + "/v1 (OpenAI-compatible)")
            print("=" * 60)

threading.Thread(target=watcher, daemon=True).start()
time.sleep(15)
if public_url:
    print("ready!")
else:
    print("timeout — check logs above")

# keep alive
try:
    while True:
        time.sleep(10)
except KeyboardInterrupt:
    print("stopping…")
    server.terminate(); proc.terminate()

## ✅ ඊට පස්සේ

| Use case | Setup |
|---|---|
| 📱 Android app | Settings ⚙️ → Server URL = public URL වන්න → Save |
| 🖥️ Web UI | URL එකට ගිහින් (`?token=...` if set) |
| 🔌 Third-party apps | Base URL `URL/v1`, key = RS_API_TOKEN, model `rs-gpt` |
| 🌐 Website | `<script src="URL/static/widget.js">` |
| 🔧 Debug | `URL/diagnose` — provider health live check |

බරපතල training එකකට (real Sinhala/English quality): repo එකේ `model/README.md` — 4B configs, Colab GPU cells.